# 📝 과제 01: 병원 예약 상담 팀 관측하기

## 만들 결과

가상 병원 '온누리병원'(수업용 가상 병원)의 **예약 상담 팀**이 같은 워커를 되풀이해 부른 원인을 궤적과 실행 기록으로 찾아 고칩니다. 슈퍼바이저가 예약 조회·진료 안내 검색·답변 작성 워커를 골라 문의를 처리하는 팀은 제공합니다. 여러분은 실행을 관측하는 코드를 쓰고, 그 코드로 원인을 찾습니다.

- Langfuse에 세션·태그를 붙여 trace를 남기는 실행 설정을 만듭니다.
- 스트리밍으로 궤적(어떤 노드가 어떤 순서로 실행됐는지의 기록)을 모으고, 노드별 실행 횟수와 반복 의심 노드를 셉니다.
- 지난주 운영 기록에서 반복의 원인을 찾아 워커 지시문을 고치고, 고친 팀으로 문의 3건을 실제로 실행해 Langfuse trace로 확인하고 `assignment01_runs.json`에 저장합니다.

교안 01(관측과 궤적 디버깅)의 방법을 적용합니다. 팀 구조는 55일차 슈퍼바이저 구조 그대로입니다.

<img src="images/assignment01_overview.png" width="1050" alt="슈퍼바이저가 예약 조회·진료 안내 검색·답변 작성 워커를 고르고, 워커는 보고를 들고 슈퍼바이저로 돌아오며, 슈퍼바이저가 끝낼 때 END로 갑니다.">

## 준비: 데이터와 제공 코드

- **자료:** `data/assignment01_*.json` 5개. 의사 7명, 의사·날짜별 예약 시각과 휴진일, 진료 안내 문서 8개(`G-01`~`G-08`), 오늘 접수된 문의 3건(`Q-301`~`Q-303`), 지난주 v1 팀이 `Q-301`을 처리한 운영 기록(`RUN_LOG`)입니다(수업용 가상 자료). 오늘은 2026-10-06(화)입니다.
- **순서:** 실행 설정 -> 궤적 모으기 -> 실행 횟수 -> 반복 판정 -> 원인 찾고 지시문 고치기 -> 실제 실행 -> 서술
- **제공:** 환경 설정, 연습용 그래프 `sample_app`, 도구·워커 지시문·워커 에이전트, 팀 State·슈퍼바이저·워커 노드·팀 그래프, `make_input`
- **작성:** `[작성]` 셀의 `...` 자리에 번호 주석대로 코드를 넣고, 바로 아래 자가채점 셀을 실행하세요. 자가채점은 모델과 Langfuse 서버를 호출하지 않습니다. 6번만 모델을 호출하고 Langfuse에 trace를 보냅니다(Langfuse 키 필요).

라이브러리·경로·키·모델 준비와 자료 읽기는 앞 일차와 같은 준비라 아래 한 셀에 모았습니다.

In [ ]:
# 환경 설정과 과제 자료: 라이브러리, 경로, API 키, 답변 모델, JSON 자료
import os
import json
from datetime import datetime
from pathlib import Path
from typing import Literal
from typing_extensions import TypedDict
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from IPython.display import Image, Markdown
# 22일차에 쓴 Langfuse입니다. CallbackHandler를 config에 넣으면 실행 한 번이 trace 하나로 남습니다.
from langfuse import get_client
from langfuse.langchain import CallbackHandler

# material_dir는 data·output이 있는 단원 폴더입니다. 정답에서는 ".."을 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 기존 값은 유지합니다.
load_dotenv(material_dir.resolve().parent / ".env")
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)
print("답변 모델:", model_name)


# JSON 읽기·저장 함수와 과제 자료입니다.
def read_json(name):
    """data 폴더의 JSON 파일을 읽어 파이썬 객체로 반환합니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """결과를 output 폴더의 JSON 파일로 저장합니다."""
    (output_dir / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


# 의사 7명: doctor_id·이름·진료과·진료 요일·비고
DOCTORS = read_json("assignment01_doctors.json")
# 휴진일 목록과 의사별·날짜별 예약 시각(시각·상태)
SLOT_DATA = read_json("assignment01_slots.json")
# 진료 안내 문서 8개: 문서 ID(G-01~G-08)·제목·내용
GUIDES = read_json("assignment01_guides.json")
# 오늘 접수된 문의 3건: 문의 ID별 고객·채널·접수 시각·본문
INQUIRIES = read_json("assignment01_inquiries.json")
# 지난주 v1 팀이 Q-301을 처리한 운영 기록: 궤적·supervisor 지시·워커 보고
RUN_LOG = read_json("assignment01_run_log.json")

# 오늘 접수된 문의 3건을 확인합니다.
for inquiry_id in INQUIRIES:
    row = INQUIRIES[inquiry_id]
    print(inquiry_id, row["채널"], row["접수 시각"], "|", row["본문"])

## 1. Langfuse에 남길 실행 설정을 만드세요

22일차에 쓴 Langfuse의 `CallbackHandler`를 config에 넣으면 그래프 실행 한 번이 trace 하나로 남습니다. 교안 01 5-2의 `make_config(session_id, thread_id, tags)`와 같은 함수를 이 팀에 맞게 만듭니다. 세션 id(`langfuse_session_id`)는 Langfuse 화면에서 trace를 묶는 이름이고, `thread_id`는 체크포인터가 State를 저장하고 다시 찾는 열쇠라서 두 값을 따로 받습니다. 태그로는 버전을 표시해 수정 후 실행의 trace를 목록에서 걸러 봅니다. 이 과제의 v1은 운영 기록(`RUN_LOG`)으로만 주어지고, Langfuse에는 6번에서 수정한 v2 실행만 남깁니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `CallbackHandler` | `langfuse.langchain`의 콜백 클래스입니다. 인자 없이 만듭니다. 세션·태그는 생성자가 아니라 config의 `metadata`로 넘깁니다. |
| `RUN_NAME` | trace 이름으로 쓸 `"hospital-team"`입니다. |
| `RECURSION_LIMIT` | 한 번 실행에서 허용할 단계 수 `30`입니다. |

**작성:** `make_config(session_id, thread_id, tags)` 함수를 완성하세요.

**목적:** 실행할 때마다 Langfuse 핸들러와 세션·태그, 단계 상한, 대화 ID를 한 번에 넘깁니다.

**구현할 처리**
1. 호출할 때마다 새 `CallbackHandler`를 만듭니다. 실행이 끝난 뒤 그 핸들러의 `last_trace_id`로 trace 주소를 찾습니다.
2. 아래 표의 다섯 키로 dict를 만듭니다.
3. dict를 반환합니다.

Langfuse 키가 없으면 핸들러를 만들 때 전송이 꺼진다는 경고가 나옵니다. 1~5번 채점에는 영향이 없습니다.

**반환값: `dict`**

| 키 | 반환할 값 |
|---|---|
| `callbacks` | 새 `CallbackHandler` 하나를 담은 리스트 |
| `run_name` | `RUN_NAME` |
| `recursion_limit` | `RECURSION_LIMIT` (`configurable` 안이 아니라 최상위 키) |
| `configurable` | `{"thread_id": thread_id}` |
| `metadata` | `{"langfuse_session_id": session_id, "langfuse_tags": tags}` |

**반환 예시:** `make_config("S-01", "S-01-Q-301", ["assignment01", "v2"])`의 결과입니다.

```python
{"callbacks": [<CallbackHandler 객체>], "run_name": "hospital-team", "recursion_limit": 30,
 "configurable": {"thread_id": "S-01-Q-301"},
 "metadata": {"langfuse_session_id": "S-01", "langfuse_tags": ["assignment01", "v2"]}}
```

In [ ]:
# 실행 이름과 단계 상한입니다.
# recursion_limit을 직접 정하지 않으면 기본값이 매우 커서 반복 호출을 막아 주지 못합니다.
RUN_NAME = "hospital-team"
RECURSION_LIMIT = 30

In [ ]:
# [작성] 실행마다 새 핸들러를 넣은 실행 설정 dict를 만드세요.
def make_config(session_id, thread_id, tags):
    """Langfuse 콜백·세션·thread_id·태그를 담은 그래프 실행 설정을 만듭니다."""
    # 1. 새 CallbackHandler를 만드세요.
    ...

    # 2. 반환값 표의 다섯 키로 config dict를 만드세요.
    ...

    # 3. config를 반환하세요.
    ...

In [ ]:
# [자가채점] config 모양만 확인합니다. Langfuse 서버에는 연결하지 않습니다.
config = make_config("S-01", "S-01-Q-301", ["assignment01", "v2"])
assert isinstance(config, dict), "dict를 반환하세요. `...`를 채웠는지도 확인하세요."
keys = sorted(config)
assert keys == ["callbacks", "configurable", "metadata", "recursion_limit", "run_name"], f"키는 callbacks, run_name, recursion_limit, configurable, metadata 다섯 개입니다. 지금 키: {keys}"
assert isinstance(config["callbacks"], list) and len(config["callbacks"]) == 1, "callbacks에는 핸들러 하나를 담은 리스트를 넣으세요."
assert isinstance(config["callbacks"][0], CallbackHandler), "callbacks의 핸들러는 CallbackHandler()로 만드세요."
assert config["run_name"] == RUN_NAME, "run_name에는 RUN_NAME을 넣으세요."
assert config["recursion_limit"] == RECURSION_LIMIT, "recursion_limit에는 RECURSION_LIMIT을 config 최상위 키로 넣으세요."
assert config["configurable"] == {"thread_id": "S-01-Q-301"}, f"configurable에는 {{'thread_id': thread_id}} 하나만 넣으세요. 인자 순서는 session_id, thread_id, tags입니다. 지금: {config['configurable']}"
assert config["metadata"] == {"langfuse_session_id": "S-01", "langfuse_tags": ["assignment01", "v2"]}, f"metadata에는 langfuse_session_id(session_id)와 langfuse_tags(tags) 두 키를 넣으세요. 지금: {config['metadata']}"
other = make_config("S-01", "S-01-Q-302", ["assignment01", "v2"])
assert other["callbacks"][0] is not config["callbacks"][0], "호출할 때마다 새 CallbackHandler를 만드세요. 함수 밖에서 하나만 만들면 실행마다 trace id를 따로 읽을 수 없습니다."
print("실행 설정 확인 완료")

## 2. 스트리밍으로 궤적을 모으세요

**궤적**은 어떤 노드가 어떤 순서로 실행됐는지의 기록입니다. 팀의 워커 노드는 안에서 에이전트(`model`, `tools` 노드)를 실행합니다. `stream`에 `stream_mode="updates"`와 `subgraphs=True`를 주면 노드 안에서 실행된 그래프의 단계까지 나오고, 이벤트마다 `(네임스페이스 튜플, {노드 이름: 그 노드가 State에 쓴 값})`이 나옵니다.

- 최상위 노드의 이벤트: `((), {"desk": {...}})`
- desk 노드 안에서 실행된 노드의 이벤트: `(("desk:1f3a...",), {"lookup": {...}})`
- `Command(goto=END)`만 반환한 노드의 이벤트: `((), {"close": None})`

네임스페이스 항목의 콜론 뒤는 LangGraph가 노드 실행 한 번마다 붙이는 작업 id라서 실행마다 달라집니다. 콜론 앞 노드 이름만 남깁니다.

| 제공 코드 | 설명 |
|---|---|
| `sample_app` | 모델 없는 연습용 그래프입니다. `desk`(안에서 `lookup` -> `answer` 실행) -> `wrap` -> `close` 순서로 실행됩니다. 입력은 `{"log": []}`입니다. |

**작성:** `collect_trajectory(app, inputs, config)` 함수를 완성하세요.

**목적:** 바깥 노드와 워커 안 에이전트의 단계를 실행 순서대로 한 목록에 모아, 실행 횟수와 반복을 셀 수 있게 합니다.

**구현할 처리**
1. 빈 리스트를 만듭니다.
2. `app`을 updates 모드로, 워커 안 단계까지 나오게 스트리밍합니다(교안 01 4절). 입력과 config는 인자로 받은 값을 쓰고, 이벤트는 `(namespace, update)`로 받습니다.
3. `namespace` 튜플의 항목마다 콜론(`:`) 앞 이름만 남기고 `"/"`로 이어 네임스페이스 문자열을 만듭니다. 빈 튜플이면 `""`입니다.
4. `update`의 노드 이름마다 `(네임스페이스 문자열, 노드 이름)` 튜플을 리스트에 더합니다. 값이 `None`인 노드도 더합니다.
5. 리스트를 반환합니다.

**반환값: `list[tuple[str, str]]`**

| 항목 | 담을 값 |
|---|---|
| 첫째 값 | 네임스페이스 문자열. 최상위 노드는 `""`, desk 안의 노드는 `"desk"` |
| 둘째 값 | 노드 이름 |

**반환 예시:** `sample_app`을 실행한 경우입니다. 안쪽 노드가 끝난 뒤에 바깥 `desk`가 끝나므로 안쪽 단계가 먼저 나옵니다.

```python
[("desk", "lookup"), ("desk", "answer"), ("", "desk"), ("", "wrap"), ("", "close")]
```

In [ ]:
# 연습용 그래프(안쪽): lookup -> answer 두 단계입니다. 모델을 부르지 않습니다.
class SampleState(TypedDict):
    log: list[str]  # 실행된 단계 이름


def lookup(state: SampleState) -> dict:
    """안쪽 그래프의 첫 단계입니다."""
    return {"log": state["log"] + ["lookup"]}


def answer(state: SampleState) -> dict:
    """안쪽 그래프의 둘째 단계입니다."""
    return {"log": state["log"] + ["answer"]}


inner_builder = StateGraph(SampleState)
inner_builder.add_node("lookup", lookup)
inner_builder.add_node("answer", answer)
inner_builder.add_edge(START, "lookup")
inner_builder.add_edge("lookup", "answer")
inner_app = inner_builder.compile()
display(Image(inner_app.get_graph().draw_mermaid_png()))


# 연습용 그래프(바깥): desk 노드 안에서 inner_app을 실행합니다.
# 워커 노드 안에서 에이전트를 실행하는 팀과 같은 모양이라, 궤적에 바깥 노드와 안쪽 노드가 함께 나와야 합니다.
def desk(state: SampleState) -> dict:
    """안쪽 그래프를 실행하고 그 기록을 돌려줍니다."""
    result = inner_app.invoke({"log": state["log"]})
    return {"log": result["log"]}


def wrap(state: SampleState) -> dict:
    """바깥 그래프의 마무리 단계입니다."""
    return {"log": state["log"] + ["wrap"]}


def close(state: SampleState) -> Command[Literal[END]]:
    """State를 바꾸지 않고 끝냅니다. 팀의 supervisor가 FINISH로 끝낼 때와 같은 모양입니다."""
    return Command(goto=END)


sample_builder = StateGraph(SampleState)
sample_builder.add_node("desk", desk)
sample_builder.add_node("wrap", wrap)
sample_builder.add_node("close", close)
sample_builder.add_edge(START, "desk")
sample_builder.add_edge("desk", "wrap")
sample_builder.add_edge("wrap", "close")
sample_app = sample_builder.compile()
display(Image(sample_app.get_graph().draw_mermaid_png()))

In [ ]:
# [작성] 스트리밍 이벤트를 (네임스페이스, 노드 이름) 목록으로 모으세요.
def collect_trajectory(app, inputs, config):
    """그래프를 스트리밍으로 실행하며 실행된 노드를 (네임스페이스, 노드 이름) 순서대로 모아 반환합니다."""
    # 1. 궤적을 담을 빈 리스트를 만드세요.
    ...

    # 2. app.stream을 updates 모드, subgraphs=True로 실행하며 이벤트마다 3~4번을 하세요.
    # 3. 네임스페이스 튜플의 항목마다 콜론(:) 앞 이름만 남겨 "/"로 이으세요. 최상위는 ""입니다.
    # 4. update의 노드 이름마다 (네임스페이스 문자열, 노드 이름) 튜플을 궤적에 더하세요.
    ...

    # 5. 궤적을 반환하세요.
    ...

In [ ]:
# [자가채점] 모델 없는 sample_app으로 궤적을 확인합니다.
trajectory = collect_trajectory(sample_app, {"log": []}, {"recursion_limit": 10})
assert isinstance(trajectory, list), "리스트를 반환하세요. `...`를 채웠는지도 확인하세요."
for item in trajectory:
    assert isinstance(item, tuple) and len(item) == 2, f"궤적의 항목은 (네임스페이스, 노드 이름) 튜플입니다. 지금 항목: {item}"
assert ("", "desk") in trajectory, f"최상위 노드의 네임스페이스는 빈 문자열 ''입니다. 지금 궤적: {trajectory}"
assert ("desk", "lookup") in trajectory, f"desk 안에서 실행된 노드가 없습니다. stream에 subgraphs=True를 넣고, 네임스페이스에서 콜론 앞 이름만 남겼는지 확인하세요. 지금 궤적: {trajectory}"
assert ("", "close") in trajectory, "반환값이 None인 노드(close)도 기록하세요. update의 노드 이름은 값과 상관없이 남깁니다."
expected = [("desk", "lookup"), ("desk", "answer"), ("", "desk"), ("", "wrap"), ("", "close")]
assert trajectory == expected, f"실행 순서대로 모으세요. 지금 궤적: {trajectory}"
print("궤적 확인 완료:", trajectory)

## 3. 노드별 실행 횟수를 세세요

궤적이 길면 눈으로 반복을 찾기 어렵습니다. 최상위 노드(슈퍼바이저와 워커)의 실행 횟수를 세면 어느 워커가 많이 불렸는지 바로 보입니다. 워커 안의 `model`, `tools`는 워커마다 같은 이름이라 세지 않습니다.

| 제공 코드 | 설명 |
|---|---|
| `RUN_LOG["trajectory"]` | 지난주 v1 팀이 `Q-301`을 처리한 궤적입니다. JSON에서 읽어 항목이 `["", "supervisor"]`처럼 두 칸 리스트입니다. |

**작성:** `count_visits(trajectory)` 함수를 완성하세요.

**목적:** 궤적에서 최상위 노드가 몇 번 실행됐는지 노드 이름별로 셉니다.

**구현할 처리**
1. 빈 dict를 만듭니다.
2. 궤적의 항목을 `(네임스페이스, 노드)` 두 값으로 풀고, 네임스페이스가 `""`인 항목만 그 노드의 횟수를 1 올립니다. 튜플과 두 칸 리스트를 모두 받습니다.
3. dict를 반환합니다.

**반환값: `dict`**

| 키 | 값 |
|---|---|
| 최상위 노드 이름 | 실행 횟수(`int`) |

**반환 예시:** supervisor -> booking_worker(안에서 model, tools) -> supervisor -> reply_writer -> supervisor 순서로 실행된 궤적입니다.

```python
{"supervisor": 3, "booking_worker": 1, "reply_writer": 1}
```

In [ ]:
# 지난주 운영 기록의 궤적 앞부분과 길이를 확인합니다. JSON에서 읽어 항목이 [네임스페이스, 노드] 리스트입니다.
print("기록한 문의:", RUN_LOG["inquiry_id"], "| 태그:", RUN_LOG["tags"], "| 처리 상한:", RUN_LOG["max_turns"])
print("궤적 길이:", len(RUN_LOG["trajectory"]))
print("앞 8개:", RUN_LOG["trajectory"][:8])

In [ ]:
# [작성] 최상위 노드만 이름별로 세세요.
def count_visits(trajectory):
    """궤적에서 최상위(네임스페이스 '') 노드가 실행된 횟수를 노드 이름별로 세어 반환합니다."""
    # 1. 횟수를 담을 빈 dict를 만드세요.
    ...

    # 2. 궤적의 (네임스페이스, 노드)마다 네임스페이스가 ""인 것만 1씩 더하세요.
    ...

    # 3. dict를 반환하세요.
    ...

In [ ]:
# [자가채점] 작은 궤적과 운영 기록으로 방문 횟수를 확인합니다.
sample = [("", "supervisor"), ("booking_worker", "model"), ("booking_worker", "tools"), ("", "booking_worker"),
          ("", "supervisor"), ("reply_writer", "model"), ("", "reply_writer"), ("", "supervisor")]
visits = count_visits(sample)
assert isinstance(visits, dict), "dict를 반환하세요. `...`를 채웠는지도 확인하세요."
assert "model" not in visits and "tools" not in visits, "에이전트 안 단계(네임스페이스가 ''이 아닌 항목)는 세지 마세요."
assert visits == {"supervisor": 3, "booking_worker": 1, "reply_writer": 1}, f"최상위 노드의 실행 횟수를 확인하세요. 지금: {visits}"
inner_named = count_visits([("desk", "lookup"), ("", "desk")])
assert inner_named == {"desk": 1}, f"노드 이름이 아니라 네임스페이스가 ''인지로 가르세요. 안쪽 노드 이름은 model·tools만 있는 것이 아닙니다. 지금: {inner_named}"
log_visits = count_visits(RUN_LOG["trajectory"])
assert log_visits == {"supervisor": 8, "booking_worker": 5, "guide_worker": 1, "reply_writer": 1}, f"JSON에서 읽은 [네임스페이스, 노드] 리스트도 같은 방법으로 세어야 합니다. 지금: {log_visits}"
print("운영 기록 방문 횟수:", log_visits)

## 4. 반복 의심 노드를 고르세요

슈퍼바이저 팀에서 같은 워커가 여러 번 불리면 MAST의 '이미 끝낸 단계를 다시 함'(교안 01 1-2 표의 대표 실패 가운데 비율이 가장 높은 실패)을 의심합니다. supervisor는 워커가 돌아올 때마다 실행되므로 횟수가 늘 많아 판정에서 뺍니다.

**작성:** `find_repeats(trajectory, limit)` 함수를 완성하세요.

**목적:** 3번의 `count_visits`로 센 횟수에서 `limit`를 넘은 워커를 골라, 궤적을 다 읽지 않고도 반복을 찾습니다.

**구현할 처리**
1. `count_visits`로 실행 횟수를 구합니다.
2. `supervisor`가 아니고 횟수가 `limit`보다 많은(초과) 노드 이름을 모읍니다. 횟수가 `limit`와 같으면 고르지 않습니다.
3. 이름순으로 정렬해 반환합니다.

**반환값: `list[str]`** 반복 의심 노드 이름의 정렬된 목록입니다. 없으면 `[]`입니다.

**반환 예시:** 운영 기록 궤적에 `limit=2`를 준 경우입니다.

```python
["booking_worker"]
```

In [ ]:
# [작성] supervisor를 뺀 노드 중 limit보다 많이 실행된 노드를 고르세요.
def find_repeats(trajectory, limit):
    """supervisor를 뺀 최상위 노드 중 실행 횟수가 limit보다 많은 노드 이름을 이름순으로 반환합니다."""
    # 1. count_visits로 노드별 실행 횟수를 구하세요.
    ...

    # 2. supervisor가 아니고 횟수가 limit보다 많은 노드 이름을 모으세요.
    ...

    # 3. 이름순으로 정렬해 반환하세요.
    ...

In [ ]:
# [자가채점] 운영 기록과 작은 궤적으로 반복 판정을 확인합니다.
repeats = find_repeats(RUN_LOG["trajectory"], 2)
assert isinstance(repeats, list), "리스트를 반환하세요. `...`를 채웠는지도 확인하세요."
assert repeats == ["booking_worker"], f"운영 기록에서 2번보다 많이 실행된 워커는 booking_worker입니다. supervisor는 빼세요. 지금: {repeats}"
assert find_repeats(RUN_LOG["trajectory"], 5) == [], "횟수가 limit와 같으면 고르지 마세요. limit보다 많은(초과) 노드만 고릅니다."
assert find_repeats(RUN_LOG["trajectory"], 0) == ["booking_worker", "guide_worker", "reply_writer"], "supervisor를 빼고, 노드 이름을 이름순으로 정렬해 반환하세요."
inner_only = [("guide_worker", "model"), ("guide_worker", "tools"), ("guide_worker", "model"), ("", "guide_worker")]
assert find_repeats(inner_only, 1) == [], "에이전트 안 단계는 세지 않습니다. count_visits로 최상위 노드만 센 횟수를 쓰세요."
print("반복 판정 확인 완료:", repeats)

## 5. 운영 기록에서 반복 원인을 찾고 지시문을 고치세요

지난주 v1 팀은 `Q-301`을 처리하며 같은 워커를 되풀이해 부르다 처리 상한(6회)에 닿았고, 고객에게 예약 가능 시간 없이 "담당자가 이어서 안내한다"는 답변을 보냈습니다. 증상(답변에 시간이 없음)에서 거꾸로 따라가 원인을 찾습니다.

<img src="images/node_focus/assignment01_booking_worker.png" width="1050" alt="팀 그래프에서 booking_worker 노드만 강조합니다.">

도구·워커 지시문·워커 에이전트는 아래 첫 셀에 모았습니다. 원인을 찾을 `BOOKING_PROMPT`는 그 셀의 워커별 지시문 부분에 있습니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `find_doctor(department)` | 진료과의 의사 목록(doctor_id·이름·진료과·진료 요일·비고) |
| `get_open_slots(doctor_id, date)` | `{"doctor_id", "날짜", "휴진", "예약 가능 시간"}` dict. 휴진일이면 `휴진`이 `True`이고 시간 목록이 비어 있습니다. |
| `search_guide(keyword)` | 제목이나 내용에 낱말이 든 진료 안내 문서 목록 |
| `BOOKING_PROMPT`, `GUIDE_PROMPT`, `REPLY_PROMPT` | 워커별 지시문 문자열(v1) |
| `booking_agent`, `guide_agent`, `reply_agent` | v1 지시문으로 만든 워커 에이전트. `name`에 변수 이름과 같은 이름을 붙였습니다(교안 01 5-1). |
| `RUN_LOG["instructions"]` | supervisor가 차례마다 워커에게 준 지시 7개 |
| `RUN_LOG["results"]` | 워커 보고 목록. 항목은 `{"worker": 워커 이름, "result": 보고}` |

**작성:** 원인을 확인하는 코드와 고친 지시문, 다시 만든 `booking_agent`를 작성하세요.

**목적:** 실행 횟수로 반복 워커를 찾고, 그 워커의 보고와 supervisor 지시를 비교해 빠진 값을 찾은 뒤, 그 값을 보고하게 지시문을 고칩니다.

**구현할 처리**
1. `find_repeats`에 운영 기록 궤적과 한도 2를 넣어 반복 의심 노드를 구하고 `repeated`에 담은 뒤, `print("반복 의심:", repeated)`로 출력합니다.
2. `RUN_LOG["results"]`에서 `worker`가 `repeated`에 든 보고의 `result` 문자열만 `repeated_reports`에 모으고, 번호를 붙여 한 줄씩 출력합니다.
3. 궤적에서 반복된 워커의 보고를, 위 셀의 supervisor 지시와 제공 코드 셀 끝의 조회 결과와 비교해 보고에서 빠진 값을 찾습니다. 그 값을 빼게 만든 `BOOKING_PROMPT`의 문장을 찾아 `CAUSE_SENTENCE`에 그대로(고치지 말고 한 문장 전체) 담습니다.
4. 아래 조건을 지키는 `BOOKING_PROMPT_FIXED`를 작성합니다.
5. `find_doctor`, `get_open_slots` 두 도구와 `BOOKING_PROMPT_FIXED`로 `booking_agent`를 다시 만듭니다. 이름은 처음과 같은 `name="booking_agent"`입니다. 6번의 팀은 이 변수를 사용합니다.

| `BOOKING_PROMPT_FIXED` 조건 | 이유 |
|---|---|
| `의사`, `날짜`, `예약 가능 시간`을 보고에 쓰게 합니다 | supervisor는 이 세 사실이 보고에 있어야 답변을 맡깁니다(처리 원칙 1). |
| 예약 가능 시간이 `없으면` 없다고, 휴진이면 휴진이라고 쓰게 합니다 | 빈 결과도 확인된 사실입니다. 보고에 없으면 supervisor가 같은 조회를 다시 맡깁니다. |
| 3에서 찾은 원인 문장(`CAUSE_SENTENCE`)을 빼고, 같은 뜻의 제한을 표현만 바꿔 남기지 않습니다 | 어느 문장인지는 직접 찾습니다. |
| 원인 문장 말고 `BOOKING_PROMPT`의 다른 문장은 고치지 않고 그대로 둡니다 | 수정 전후 차이가 원인 문장 하나여야, 반복이 줄어든 것을 그 문장을 고친 효과로 확인할 수 있습니다. |

**반환값:** 아래 변수 5개를 만듭니다.

| 변수 | 담을 값 |
|---|---|
| `repeated` | 반복 의심 노드 이름 목록 |
| `repeated_reports` | 반복된 워커의 보고(`result` 문자열) 목록. 운영 기록 순서 그대로 |
| `CAUSE_SENTENCE` | 반복의 원인으로 찾은 `BOOKING_PROMPT` 안의 문장(그대로 복사한 문자열) |
| `BOOKING_PROMPT_FIXED` | 고친 지시문 문자열 |
| `booking_agent` | 다시 만든 예약 조회 에이전트(이름 `booking_agent`) |

**반환 예시:** 1~2의 출력 모양입니다.

```text
반복 의심: ['booking_worker']
1. 확인한 의사: 이도윤(정형외과)
2. ...
```

In [ ]:
# 예약 조회·진료 안내 검색 도구입니다. 조회만 하고 예약을 바꾸지 않습니다.
@tool
def find_doctor(department: str) -> list[dict]:
    """진료과 이름(예: '정형외과')으로 의사 목록을 찾습니다. doctor_id·이름·진료과·진료 요일·비고가 있습니다."""
    found = []
    for doctor in DOCTORS:
        if doctor["진료과"] == department:
            found.append(doctor)

    return found


@tool
def get_open_slots(doctor_id: str, date: str) -> dict:
    """의사(doctor_id)의 날짜(YYYY-MM-DD) 예약 가능 시각을 조회합니다. 휴진일이면 휴진이 True이고 시각 목록이 비어 있습니다."""
    # 예약 시각이 등록되지 않은 날짜는 빈 목록으로 봅니다.
    day_slots = SLOT_DATA["slots"].get(doctor_id, {}).get(date, [])
    times = []
    for slot in day_slots:
        if slot["상태"] == "예약 가능":
            times.append(slot["시각"])

    return {"doctor_id": doctor_id, "날짜": date, "휴진": date in SLOT_DATA["휴진일"], "예약 가능 시간": times}


@tool
def search_guide(keyword: str) -> list[dict]:
    """제목이나 내용에 keyword가 들어간 진료 안내 문서를 찾습니다. keyword는 '금식'처럼 짧은 낱말 하나입니다."""
    found = []
    for guide in GUIDES:
        if keyword in guide["제목"] or keyword in guide["내용"]:
            found.append(guide)

    return found


# 워커별 지시문입니다. 오늘은 2026-10-06(화)입니다.
BOOKING_PROMPT = (
    "온누리병원 예약 조회 담당입니다. 오늘은 2026-10-06(화)입니다. 지시받은 진료과의 의사를 find_doctor로 찾고, "
    "get_open_slots로 지시받은 날짜의 예약 시각을 조회하세요. 날짜는 2026-10-08처럼 씁니다. "
    "보고는 '확인한 의사: 이름(진료과)' 한 줄로만 쓰세요.")
GUIDE_PROMPT = (
    "온누리병원 진료 안내 담당입니다. search_guide는 낱말이 들어 있는지만 보므로 '금식', '휴진', '초진'처럼 "
    "짧은 낱말 하나로 검색하세요. 찾은 문서의 문서 ID와 필요한 내용을 짧게 보고하고, 못 찾으면 다른 낱말로 다시 검색하세요.")
REPLY_PROMPT = (
    "온누리병원 예약 상담 답변 담당입니다. 전달받은 보고에 있는 사실만으로 고객에게 보낼 한국어 답변을 쓰세요. "
    "예약 가능 시각은 보고에 적힌 09:30 같은 형식 그대로 쓰세요. 준비물·금식 같은 진료 안내는 guide_worker가 "
    "문서 ID와 함께 보고한 내용만 쓰고, 그 문서 ID를 근거로 밝히세요. "
    "보고에 없는 시간이나 예약 확정을 쓰지 마세요.")

# 워커 에이전트입니다. 워커마다 필요한 도구만 줍니다. 답변 담당은 보고만 읽고 쓰므로 도구가 없습니다.
# name: 이름에 agent가 들어가야 Langfuse가 agent로 표시합니다(교안 01 5-1).
booking_agent = create_agent(llm, tools=[find_doctor, get_open_slots], name="booking_agent", system_prompt=BOOKING_PROMPT)
guide_agent = create_agent(llm, tools=[search_guide], name="guide_agent", system_prompt=GUIDE_PROMPT)
reply_agent = create_agent(llm, tools=[], name="reply_agent", system_prompt=REPLY_PROMPT)

# 도구가 돌려주는 값을 확인합니다.
print("조회 결과:", get_open_slots.invoke({"doctor_id": "D-11", "date": "2026-10-08"}))

In [ ]:
# 운영 기록에서 supervisor가 워커에게 준 지시를 순서대로 읽습니다.
# 지시 i번은 그 차례에 일을 맡은 워커에게 간 것입니다.
for index, instruction in enumerate(RUN_LOG["instructions"], start=1):
    print(f"{index}.", instruction)

In [ ]:
# [작성] 운영 기록에서 반복된 워커와 그 보고를 확인하고, 지시문을 고쳐 에이전트를 다시 만드세요.
# 1. find_repeats로 운영 기록 궤적에서 2번보다 많이 실행된 노드를 찾아 repeated에 담고 출력하세요.
...

# 2. RUN_LOG["results"]에서 worker가 repeated에 든 보고의 result만 repeated_reports에 모으고 번호를 붙여 출력하세요.
...

# 3. 지시·보고·도구 조회 결과를 비교해, 반복의 원인이 된 BOOKING_PROMPT의 문장을 CAUSE_SENTENCE에 그대로 담으세요.
...

# 4. 원인 문장만 빼고 나머지 문장은 그대로 둔 채, 보고에 빠진 값을 반드시 쓰게 하는 BOOKING_PROMPT_FIXED를 작성하세요.
...

# 5. 같은 두 도구, 같은 이름(name), BOOKING_PROMPT_FIXED로 booking_agent를 다시 만드세요.
...

In [ ]:
# [자가채점] 원인 확인, 고친 지시문, 다시 만든 에이전트를 확인합니다. 모델은 고정 응답으로 바꿔 호출하지 않습니다.
from unittest.mock import Mock, patch
from langchain_core.messages import AIMessage

assert "repeated" in globals() and repeated == ["booking_worker"], "find_repeats로 운영 기록 궤적에서 2번보다 많이 실행된 노드를 구해 repeated에 담으세요."
assert "repeated_reports" in globals() and isinstance(repeated_reports, list), "repeated_reports 리스트를 만드세요."
assert len(repeated_reports) == 5, f"RUN_LOG['results']에서 worker가 repeated에 든 보고만 모으세요. 지금 {len(repeated_reports)}개입니다."
for report in repeated_reports:
    assert isinstance(report, str), "repeated_reports에는 보고 dict가 아니라 result 문자열을 담으세요."
assert "CAUSE_SENTENCE" in globals() and isinstance(CAUSE_SENTENCE, str), "CAUSE_SENTENCE에 원인으로 찾은 문장을 담으세요."
prompt_sentences = [part.strip() + "." for part in BOOKING_PROMPT.split(".") if part.strip()]
assert CAUSE_SENTENCE.strip() in prompt_sentences, "CAUSE_SENTENCE에는 BOOKING_PROMPT 안의 문장 하나를 마침표까지 고치지 말고 그대로 담으세요. 문장 일부만 담으면 통과하지 못합니다."
assert "한 줄" in CAUSE_SENTENCE, "CAUSE_SENTENCE가 반복의 원인 문장이 아닙니다. 보고에 무엇이 빠졌는지, BOOKING_PROMPT의 어느 문장이 보고 내용을 정하는지 다시 비교하세요."
assert "BOOKING_PROMPT_FIXED" in globals() and isinstance(BOOKING_PROMPT_FIXED, str), "BOOKING_PROMPT_FIXED 문자열을 만드세요."
for word in ["의사", "날짜", "예약 가능 시간", "없으면"]:
    assert word in BOOKING_PROMPT_FIXED, f"BOOKING_PROMPT_FIXED에 '{word}'가 들어가야 합니다. 보고에 쓸 항목과 결과가 없을 때 쓸 내용을 적으세요."
assert CAUSE_SENTENCE not in BOOKING_PROMPT_FIXED, "원인으로 찾은 문장이 고친 지시문에 그대로 남아 있습니다."
for sentence in prompt_sentences:
    if sentence != CAUSE_SENTENCE.strip():
        assert sentence in BOOKING_PROMPT_FIXED, f"원인 문장 말고 BOOKING_PROMPT의 다른 문장은 고치지 말고 그대로 두세요. 빠진 문장: {sentence}"
for limit in ["한 줄", "한줄", "로만", "이내", "한 문장"]:
    assert limit not in BOOKING_PROMPT_FIXED, f"고친 지시문에 보고 길이나 내용을 줄이는 제한('{limit}')이 남아 있습니다. 원인 문장을 표현만 바꿔 남기지 말고 빼세요."
assert "tools" in booking_agent.nodes, "booking_agent를 create_agent로 다시 만들고 도구를 주세요."
names = sorted(booking_agent.nodes["tools"].bound.tools_by_name)
assert names == ["find_doctor", "get_open_slots"], f"booking_agent의 도구는 find_doctor, get_open_slots 그대로입니다. 지금: {names}"
assert booking_agent.name == "booking_agent", f"booking_agent를 다시 만들 때 이름(name)도 처음과 같이 주세요. 지금 이름: {booking_agent.name}"
fake = Mock(return_value=AIMessage(content="채점용 보고"))
with patch.object(type(llm), "invoke", fake):
    booking_agent.invoke({"messages": [{"role": "user", "content": "D-11의 2026-10-08 예약 가능 시간을 보고하세요."}]})
system_message = fake.call_args.args[0][0]
assert system_message.content == BOOKING_PROMPT_FIXED, "booking_agent를 system_prompt=BOOKING_PROMPT_FIXED로 다시 만드세요. 지금 에이전트는 다른 지시문을 씁니다."
print("원인과 지시문 확인 완료")

## 6. 고친 팀으로 문의 3건을 실제로 실행하세요

고친 `booking_agent`로 팀을 실행해 반복이 줄었는지 궤적으로 확인하고, Langfuse에 같은 세션·태그로 trace를 남깁니다. 이 문항에서만 모델을 호출합니다.

교안 01은 작업 하나 = thread 하나 = 세션 하나로 두었습니다. 이 과제는 교안 01 5-2의 **다르게 두는 경우**를 써서, 세션은 오늘 접수된 문의 묶음 하나(`SESSION_ID`), thread는 문의 하나(`f"{SESSION_ID}-{inquiry_id}"`)로 둡니다. 수정 후 실행 3건을 Langfuse 세션 화면 하나에서 나란히 보기 위해서이며, 문의마다 State는 따로 저장됩니다.

팀 State·슈퍼바이저·워커 노드·`make_input`·그래프 조립은 55일차 구조 그대로라 아래 한 셀에 모았습니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `team_app` | 슈퍼바이저와 워커 셋의 팀 그래프입니다. 체크포인터가 있어 `team_app.get_state(config).values`로 끝난 State를 읽습니다. |
| `make_input(inquiry_id)` | 문의 ID를 받아 팀의 처음 State dict를 반환합니다. 처리 상한은 6회입니다. |
| `langfuse` | `get_client()`로 만든 Langfuse 클라이언트입니다. `get_trace_url(trace_id=...)`, `flush()`를 씁니다. |
| `SESSION_ID`, `TAGS` | 오늘 문의 묶음의 세션 ID(`assignment01-실행 시각`)와 태그 `["assignment01", "v2-fixed-prompt"]`입니다. |

**작성:** 문의 3건을 실행하고 `runs`에 결과를 담아 저장하세요.

**목적:** 1~4번에서 만든 관측 함수로 실제 팀의 궤적을 모으고, 같은 세션의 trace 3개를 Langfuse에서 볼 수 있게 남깁니다.

**구현할 처리**
1. 결과를 담을 `runs` dict를 만듭니다.
2. 문의마다 `make_config`로 실행 설정을 만듭니다. 세션은 `SESSION_ID`, `thread_id`는 `f"{SESSION_ID}-{inquiry_id}"`, 태그는 `TAGS`입니다.
3. `collect_trajectory`로 `team_app`을 실행해 궤적을 받습니다. 입력은 `make_input(inquiry_id)`입니다.
4. 같은 config로 `get_state`를 읽어 마지막 보고(`results`의 마지막 항목의 `result`)를 답변으로 꺼내고, config의 핸들러 `last_trace_id`로 trace 주소를 만듭니다.
5. `runs[inquiry_id]`에 아래 네 키를 담고 실행 횟수를 출력합니다.
6. `langfuse.flush()`로 전송을 마치고 `save_json("assignment01_runs.json", runs)`로 저장합니다.

**반환값: `dict`** `runs[inquiry_id]`는 아래 네 키를 가진 dict입니다.

| 키 | 담을 값 |
|---|---|
| `trajectory` | `collect_trajectory`가 돌려준 궤적 |
| `visits` | 그 궤적의 `count_visits` 결과 |
| `reply` | 고객 답변(마지막 보고의 `result`) |
| `trace_url` | `langfuse.get_trace_url(trace_id=handler.last_trace_id)`로 얻은 trace 주소 |

**반환 예시:** Q-303을 실행한 경우입니다.

```python
{"Q-303": {"trajectory": [("", "supervisor"), ...],
           "visits": {"supervisor": ..., "reply_writer": ..., ...},
           "reply": "안녕하세요. ...", "trace_url": "https://.../project/.../traces/..."}}
```

경로와 방문 수는 모델 판단에 따라 실행마다 다릅니다.

**확인 기준:** 자가채점이 저장된 파일로 아래를 확인합니다.
- 세 문의 모두 키가 `trajectory`, `visits`, `reply`, `trace_url`이고, `visits`가 그 궤적의 `count_visits` 결과와 같습니다.
- 체크포인터에 문의마다 `f"{SESSION_ID}-{inquiry_id}"` thread의 State가 있고, `reply`가 그 State의 마지막 보고와 같습니다.
- Q-301·Q-302 궤적에 워커 안 에이전트의 단계 `["booking_worker", "tools"]`가 있고, 세 문의 모두 마지막으로 일한 워커가 `reply_writer`입니다. Q-303은 휴진 안내 문서만으로 답할 수도 있어 예약 조회를 요구하지 않습니다.
- `find_repeats(궤적, 2)`가 `[]`입니다(어느 워커도 2번보다 많이 불리지 않음).
- Q-301 답변에 10월 8일 예약 가능 시각 `09:30` 또는 `14:00`이 있고, Q-303 답변에 `휴진` 안내가 있습니다.

모델 응답을 고정하거나 결과를 고쳐 쓰지 마세요. 경로와 문장은 모델 판단에 따라 달라질 수 있으니 실패하면 다시 실행해 보세요.

In [ ]:
# 팀 State와 슈퍼바이저가 쓰는 팀 소개, 처리 원칙, 구조화 출력 체인입니다(55일차 슈퍼바이저 구조 그대로).
class TeamState(TypedDict):
    inquiry: str  # 문의 정보(문의 ID·고객·채널·접수 시각·본문)
    max_turns: int  # 처리 상한: 워커에게 일을 맡길 수 있는 횟수. 닿으면 reply_writer에게 1번 더 맡깁니다
    instruction: str  # 이번 워커에게 준 지시
    visits: list[str]  # 일을 맡긴 워커 순서
    turn: int  # 일을 맡긴 횟수
    results: list[dict]  # [{"worker": 워커 이름, "result": 보고}, ...]


WORKER_GUIDE = (
    "booking_worker: 진료과 의사 찾기(find_doctor)와 의사·날짜별 예약 가능 시간 조회(get_open_slots). "
    "guide_worker: 진료 안내 문서 검색(준비물·금식·휴진·예방접종 등). "
    "reply_writer: 보고된 사실로 고객 답변 작성."
)

# 처리 원칙: 워커를 부르는 순서는 정하지 않고 답변을 맡기기 전에 갖춰야 할 기준만 적습니다.
TEAM_RULES = (
    "처리 원칙: "
    "1) 예약 문의에 답하려면 의사, 날짜, 그 날짜의 예약 가능 시간(없으면 없다는 사실과 휴진 여부)이 보고로 확인되어야 합니다. "
    "2) 판단에 필요한 사실이 아직 보고에 없으면 그 사실을 알아낼 수 있는 워커에게 맡깁니다. 이미 보고된 내용은 다시 시키지 않습니다. "
    "3) 준비물·금식·휴진 같은 안내는 guide_worker가 보고한 문서 ID를 근거로 합니다. "
    "4) 고객 답변에 필요한 사실이 모두 보고되면 reply_writer에게 답변을 맡기고, 답변이 나오면 FINISH를 고릅니다."
)

# 처리 상한에 닿았을 때 reply_writer에게 주는 지시입니다.
CAP_INSTRUCTION = (
    "처리 상한에 닿았습니다. 보고에 없는 내용을 확정하지 말고, 지금까지 확인한 내용과 함께 "
    "'예약센터 담당자가 이어서 안내한다'고 쓰세요."
)


# next는 그래프의 노드 이름과 같아야 그대로 goto에 쓸 수 있습니다.
class Route(BaseModel):
    next: Literal["booking_worker", "guide_worker", "reply_writer", "FINISH"] = Field(
        description="다음에 일을 맡길 워커. reply_writer가 답변을 썼으면 FINISH")
    instruction: str = Field(description="다음 워커에게 줄 지시. 진료과·의사·날짜 같은 필요한 값을 넣는다. FINISH이면 빈 문자열")


supervisor_prompt = ChatPromptTemplate.from_messages([
    ("system", "온누리병원 예약 상담 팀의 팀장입니다. 오늘은 2026-10-06(화)입니다. 직접 조회하지 않고 워커에게 맡깁니다. "
     + WORKER_GUIDE + " " + TEAM_RULES + " 문의와 지금까지 보고를 읽고 다음 워커 한 명과 지시를 정하세요."),
    ("human", "문의:\n{inquiry}\n\n지금까지 보고:\n{results}"),
])
supervisor_chain = supervisor_prompt | llm.with_structured_output(Route, strict=True)


def format_results(results):
    """워커 보고 목록을 '[워커] 보고' 문단으로 묶습니다."""
    # 아직 보고가 없으면 첫 판단이라는 뜻으로 '아직 없음'을 넘깁니다.
    if not results:
        return "아직 없음"

    lines = []
    for row in results:
        lines.append(f"[{row['worker']}] {row['result']}")

    return "\n\n".join(lines)


# 슈퍼바이저와 워커 노드입니다. 이동은 모두 Command가 정합니다.
def supervisor(state: TeamState) -> Command[Literal["booking_worker", "guide_worker", "reply_writer", END]]:
    """처리 상한을 지키며 다음 워커와 지시를 정해 Command로 보냅니다."""
    # 상한에 닿으면 모델을 부르지 않고 답변으로 마무리합니다.
    if state["turn"] >= state["max_turns"]:
        if "reply_writer" in state["visits"]:
            return Command(goto=END)
        return Command(goto="reply_writer", update={
            "instruction": CAP_INSTRUCTION,
            "visits": state["visits"] + ["reply_writer"],
            "turn": state["turn"] + 1})

    route = supervisor_chain.invoke({"inquiry": state["inquiry"], "results": format_results(state["results"])})
    if route.next == "FINISH":
        return Command(goto=END)

    return Command(goto=route.next, update={
        "instruction": route.instruction,
        "visits": state["visits"] + [route.next],
        "turn": state["turn"] + 1})


def run_worker(agent, worker, state):
    """문의·지시·지금까지 보고를 워커 에이전트에 전달하고, 새 보고를 덧붙인 results를 반환합니다."""
    # 마지막 줄은 모든 워커에 주는 역할 경계입니다. 다른 담당이 맡을 내용을 지어 보고하지 않게 합니다.
    message = (f"문의:\n{state['inquiry']}\n\n지시: {state['instruction']}\n\n"
               f"팀이 지금까지 확인한 내용:\n{format_results(state['results'])}\n\n"
               "지시받은 일의 결과만 보고하세요. 지시에 없는 다른 담당의 일(예: 준비물·금식 안내)은 보고에 쓰지 않습니다.")
    reply = agent.invoke({"messages": [{"role": "user", "content": message}]})

    # 마지막 메시지(워커의 최종 보고)만 기존 보고 뒤에 붙입니다.
    return state["results"] + [{"worker": worker, "result": reply["messages"][-1].text}]


def booking_worker(state: TeamState) -> Command[Literal["supervisor"]]:
    """booking_agent의 보고를 results에 더하고 supervisor로 돌아갑니다."""
    # 실행할 때 전역 booking_agent를 읽으므로 5번에서 다시 만든 에이전트가 쓰입니다.
    results = run_worker(booking_agent, "booking_worker", state)
    return Command(goto="supervisor", update={"results": results})


def guide_worker(state: TeamState) -> Command[Literal["supervisor"]]:
    """guide_agent의 보고를 results에 더하고 supervisor로 돌아갑니다."""
    results = run_worker(guide_agent, "guide_worker", state)
    return Command(goto="supervisor", update={"results": results})


def reply_writer(state: TeamState) -> Command[Literal["supervisor"]]:
    """reply_agent의 답변을 results에 더하고 supervisor로 돌아갑니다."""
    results = run_worker(reply_agent, "reply_writer", state)
    return Command(goto="supervisor", update={"results": results})


# 문의 한 건으로 팀의 처음 State를 만듭니다. 보고·방문 기록은 비우고 턴은 0부터 셉니다.
def make_input(inquiry_id, max_turns=6):
    """문의 ID를 받아 문의 정보 문자열과 빈 기록을 담은 처음 State를 반환합니다."""
    row = INQUIRIES[inquiry_id]
    inquiry = (f"문의 ID: {inquiry_id}\n고객: {row['고객']}\n채널: {row['채널']}\n"
               f"접수 시각: {row['접수 시각']}\n본문: {row['본문']}")

    return {"inquiry": inquiry, "max_turns": max_turns, "instruction": "", "visits": [], "turn": 0, "results": []}


# 팀 그래프입니다. 체크포인터를 붙여 실행이 끝난 뒤 get_state(config)로 마지막 State를 읽을 수 있게 합니다.
team_builder = StateGraph(TeamState)
team_builder.add_node("supervisor", supervisor)
team_builder.add_node("booking_worker", booking_worker)
team_builder.add_node("guide_worker", guide_worker)
team_builder.add_node("reply_writer", reply_writer)
team_builder.add_edge(START, "supervisor")
team_app = team_builder.compile(checkpointer=InMemorySaver())
display(Image(team_app.get_graph().draw_mermaid_png()))

In [ ]:
# Langfuse 연결을 확인하고 이번 실행의 세션 ID와 태그를 정합니다. 이 셀부터 Langfuse 키가 필요합니다.
langfuse = get_client()
print("Langfuse 연결:", langfuse.auth_check())

# 세션 ID는 오늘 문의 묶음의 이름입니다. 실행 시각을 넣어 다시 실행하면 새 세션이 되게 합니다. 태그로 수정 후 실행(v2)을 표시합니다.
SESSION_ID = f"assignment01-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
TAGS = ["assignment01", "v2-fixed-prompt"]
print("세션:", SESSION_ID, "| 태그:", TAGS)

In [ ]:
# [작성] 문의 3건을 같은 세션으로 실행하고 궤적과 trace 주소를 저장하세요.
# 1. 결과를 담을 runs dict를 만드세요.
runs = {}
for inquiry_id in INQUIRIES:
    # 2. 세션 SESSION_ID, 문의마다 다른 thread_id, 태그 TAGS로 실행 설정을 만드세요.
    ...

    # 3. collect_trajectory로 team_app을 실행하고 궤적을 받으세요.
    ...

    # 4. 체크포인터에 남은 마지막 State에서 답변을 꺼내고, 핸들러의 trace id로 trace 주소를 만드세요.
    ...

    # 5. runs[inquiry_id]에 trajectory·visits·reply·trace_url을 담고 방문 횟수를 출력하세요.
    ...

# 6. 전송을 마치고 runs를 assignment01_runs.json으로 저장하세요.
...

In [ ]:
# [자가채점] 저장된 실행 기록으로 확인합니다. 모델과 Langfuse 서버는 호출하지 않습니다.
import re

path = output_dir / "assignment01_runs.json"
assert path.exists(), "save_json으로 runs를 assignment01_runs.json에 저장하세요."
saved = json.loads(path.read_text(encoding="utf-8"))
RETRY_HINT = "코드가 맞다면 모델 판단이 달라진 것일 수 있습니다. 실행 셀과 이 셀을 다시 실행해 보세요."
for inquiry_id in INQUIRIES:
    assert inquiry_id in saved, f"{inquiry_id} 결과가 없습니다. 문의 3건을 모두 실행하세요."
    run = saved[inquiry_id]
    keys = sorted(run)
    assert keys == ["reply", "trace_url", "trajectory", "visits"], f"{inquiry_id}: 키는 trajectory, visits, reply, trace_url입니다. 지금 키: {keys}"
    assert run["visits"] == count_visits(run["trajectory"]), f"{inquiry_id}: visits에는 그 실행 궤적의 count_visits 결과를 담으세요."
    if inquiry_id != "Q-303":
        assert ["booking_worker", "tools"] in run["trajectory"], f"{inquiry_id}: 궤적에 booking_worker 안의 도구 단계가 없습니다. collect_trajectory가 subgraphs=True로 모으는지 확인하세요. {RETRY_HINT}"
    workers = []
    for namespace, node in run["trajectory"]:
        if namespace == "" and node != "supervisor":
            workers.append(node)
    assert workers and workers[-1] == "reply_writer", f"{inquiry_id}: 마지막으로 일한 워커는 reply_writer여야 합니다. {RETRY_HINT}"
    repeats = find_repeats(run["trajectory"], 2)
    assert repeats == [], f"{inquiry_id}: 2번보다 많이 실행된 워커가 있습니다: {repeats}. BOOKING_PROMPT_FIXED가 예약 가능 시간을 보고하게 하는지 확인하세요. {RETRY_HINT}"
    assert run["reply"].strip() != "", f"{inquiry_id}: reply에 마지막 보고(고객 답변)를 담으세요."
    langfuse_host = os.getenv("LANGFUSE_BASE_URL", "https://cloud.langfuse.com").rstrip("/")
    assert run["trace_url"].startswith(langfuse_host) and re.search(r"/traces/[0-9a-f]{32}$", run["trace_url"]), f"{inquiry_id}: trace_url에는 langfuse.get_trace_url로 얻은 이번 실행의 trace 주소를 담으세요."
for inquiry_id in INQUIRIES:
    thread = {"configurable": {"thread_id": f"{SESSION_ID}-{inquiry_id}"}}
    values = team_app.get_state(thread).values
    assert values, f"{inquiry_id}: 체크포인터에 thread_id '{SESSION_ID}-{inquiry_id}'의 State가 없습니다. thread_id는 문의마다 SESSION_ID와 문의 ID를 '-'로 이어 두세요."
    assert saved[inquiry_id]["reply"] == values["results"][-1]["result"], f"{inquiry_id}: reply에는 끝난 State의 results 마지막 항목의 result를 담으세요."
first_reply = saved["Q-301"]["reply"]
assert "09:30" in first_reply or "14:00" in first_reply, f"Q-301 답변에 10월 8일 예약 가능 시각(09:30, 14:00)이 없습니다. {RETRY_HINT}"
assert "휴진" in saved["Q-303"]["reply"], f"Q-303 답변에 10월 9일 휴진 안내가 없습니다. {RETRY_HINT}"
print("실제 실행 기록 확인 완료")

Q-301 고객 답변입니다. 10월 8일 예약 가능 시각과 초진 준비물(문서 ID)을 보고에 있는 그대로 안내하는지 읽어 보세요.

In [ ]:
# Q-301에 대한 고객 답변을 읽어 봅니다.
display(Markdown(runs["Q-301"]["reply"]))

## 7. 반복의 원인을 무엇으로 판단했는지 설명하세요

v1 실행은 Langfuse trace 없이 운영 기록(`RUN_LOG`)으로만 남아 있습니다. 6번에서 남긴 v2 세션은 Langfuse에서 열 수 있습니다.

**작성:** 서술형 답안

**목적:** 궤적·실행 횟수 같은 숫자와 관측 화면의 어느 부분이 원인 판단의 근거가 되는지 설명합니다.

**설명할 내용**
1. 운영 기록(`RUN_LOG`)에서 반복과 원인을 판단한 근거를 쓰세요. 실행 횟수, supervisor 지시, `booking_worker` 보고, `get_open_slots` 조회 결과 가운데 무엇을 비교했습니까?
2. v1을 Langfuse로 남겼다면 같은 신호가 어느 화면에서 보일지 쓰세요. trace 상세의 **Graph** 탭에 나오는 실행 횟수와, supervisor 아래 모델 호출(ChatOpenAI)의 입력을 중심으로 답하세요.
3. 6번 v2 세션의 trace에서 실제로 확인한 것을 쓰세요. 문의마다 `booking_worker` 방문 수와 답변에 들어간 값(예약 가능 시각, 휴진)을 적으세요.

정답 노트북의 모범 설명과 비교하세요.

*(여기에 확인한 화면과 판단 근거를 작성하세요.)*